# Partie 2, session 13 : 2WikiMultiHopQA, modèle de départ (graines de génération 0 et 1) (version Colab, un GPU)

Amendement du 24 novembre : le modèle de départ tire 32 essais avec la graine 0 (paire principale) et 32 avec la graine 1 (64 au total, pour les essais équivalents) ; moitiés sur deux GPU.

Avant de lancer : **Exécution > Modifier le type d'exécution > GPU T4**. Garder l'onglet ouvert (la version gratuite coupe une session inactive). Si la session est coupée : relancer toutes les cellules, les runs reprennent là où ils se sont arrêtés.

Résultats : `MyDrive/memoire-rl-search/s13/`. Notebook généré par `scripts/genere_colab.py`.


In [ ]:
BRANCH = "partie2-passk"
N_QUESTIONS, N_SAMPLES, T, TEMP = 500, 32, 4, 0.7   # valeurs par défaut ; les options d'un run priment
MAX_TOKENS, MAX_MODEL_LEN = 1024, 8192   # figés au pilote v2
SESSION = 13
RUNS = [
    ('Qwen/Qwen2.5-3B-Instruct', ['--type', 'compositional', '--data', '']),
    ('Qwen/Qwen2.5-3B-Instruct', ['--type', 'compositional', '--data', '', '--seed', '1']),
]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK = Path(f'/content/drive/MyDrive/memoire-rl-search/s{SESSION}')
(WORK / 'logs').mkdir(parents=True, exist_ok=True)
print(WORK)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/content/memoire-rl-search")
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q vllm==0.33.0 pandas pyarrow huggingface_hub 2>&1 | tail -3
!pip freeze > {WORK}/pip_freeze.txt
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"
import sys
sys.path.insert(0, str(CLONE))
from data.hotpot_distractor import telecharger
PARQUET = telecharger("/content/hotpot")


In [ ]:
import os, time, json
status = []
env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD='spawn')
for model, opts in RUNS:
    nom = model.split('/')[-1] + ''.join('_' + o.strip('-').replace('/', 'of') for o in opts if o)
    cmd = ['python', '-u', 'scripts/run_passk.py', '--model', model, '--type', 'bridge',
           '--n-questions', str(N_QUESTIONS), '--n-samples', str(N_SAMPLES), '--max-turns', str(T),
           '--temperature', str(TEMP), '--data', PARQUET, '--max-tokens', str(MAX_TOKENS),
           '--max-model-len', str(MAX_MODEL_LEN), '--outdir', str(WORK / 'results' / 'partie2')] + opts
    t0 = time.time()
    with open(WORK / 'logs' / f'{nom}.log', 'a') as f:
        code = subprocess.run(cmd, cwd=CLONE, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
    status.append((nom, 0, code, round(time.time() - t0)))
    print(f'{nom} : code {code}, {time.time() - t0:.0f} s', flush=True)
    if code != 0:
        print(open(WORK / 'logs' / f'{nom}.log').read()[-3000:])


In [ ]:
from collections import Counter
resume = {}
for fichier in sorted((WORK / 'results' / 'partie2').glob('*.jsonl')):
    L = [json.loads(l) for l in fichier.read_text().split('\n') if l.strip()]
    if not L:
        continue
    par_q = Counter(l['id'] for l in L)
    resume[fichier.stem] = {'trajectoires': len(L),
        'questions_completes': sum(v >= N_SAMPLES for v in par_q.values()),
        'em': round(sum(l['em'] for l in L) / len(L), 3), 'statuts': dict(Counter(l['status'] for l in L)),
        'tronquees': round(sum(l['truncated'] for l in L) / len(L), 3),
        'recherches': round(sum(l['n_search'] for l in L) / len(L), 2),
        'jetons': round(sum(l['gen_tokens'] for l in L) / len(L))}
(WORK / f'resume_s{SESSION}.json').write_text(json.dumps({'status': status, 'resume': resume}, indent=1, ensure_ascii=False))
print(json.dumps({'status': status, 'resume': resume}, indent=1, ensure_ascii=False))
echecs = [s for s in status if s[2] != 0]
if echecs:
    raise RuntimeError(f'Runs en échec : {echecs}')
